In [ ]:
# Kaggle setup (does nothing outside Kaggle). Needs: GPU on, Internet on, competition data as input.
# The repo stays private until the deadline, so it is cloned with a GitHub token saved as a
# Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets; fine-grained token, read-only, this repo only).
import os, subprocess
REPO = "polluvl/hw2-video-detection"   # <- set once, e.g. "polluvl/hw2-video-detection"
if os.path.isdir("/kaggle/working") and not os.path.isdir("/kaggle/working/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        auth = ""  # no secret -> works once the repo is public
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
                       capture_output=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: check REPO and the GITHUB_TOKEN secret")
    # remove the token from the saved clone (outputs of /kaggle/working can be shared)
    subprocess.run(["git", "-C", "/kaggle/working/repo", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
if os.path.isdir("/kaggle/working/repo/notebooks"):
    os.chdir("/kaggle/working/repo/notebooks")

# 03 · One-stage anchor-based: RetinaNet (torchvision, ResNet-50-FPN v2, COCO-pretrained)
Same protocol as 02 (same frames, resize, augmentation, optimiser, schedule, evaluator).

Default torchvision anchors: sizes 32–512 px on P3–P7 (3 scales × ratios 0.5/1/2).
The EDA showed most objects are much smaller than 32 px after resizing, so the tuned runs use
anchors fitted on the TRAIN split (`anchors.json`, written by 01_eda).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from analysis import (draw_errors, error_breakdown, recall_by_size,
                      summarise_errors, worst_frames)
from common import (SMOKE, evaluate, find_data_root, list_frames, log_result,
                    save_preds, seed_everything, work_dir)
from tv_detection import build_retinanet, predict, train_detector

DATA = find_data_root()
OUT = work_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_STEP, VAL_STEP, EPOCHS = (1, 1, 1) if SMOKE else (10, 5, 8)
train_frames = list_frames(DATA / "train", step=TRAIN_STEP)
val_frames = list_frames(DATA / "val", step=VAL_STEP)

BASE = dict(epochs=EPOCHS, batch_size=2 if SMOKE else 4, lr=0.01, num_workers=2,
            min_size=800, max_size=1333)
if SMOKE:
    BASE.update(min_size=192, max_size=320, print_every=5)

with open(OUT / "anchors.json") as f:
    anchors = json.load(f)
print("data-driven anchors:", anchors)
DATA_ANCHORS = {"anchor_sizes": anchors["retina_anchor_sizes"], "aspect_ratios": anchors["aspect_ratios"]}

## Experiments (one change at a time)
- `retina_base`     – torchvision defaults
- `retina_anchors`  – anchor sizes/ratios from the training-set box statistics
- `retina_anchors_iou` – + lower IoU thresholds for positives/negatives (0.4 / 0.3 instead of
  0.5 / 0.4): tiny boxes rarely reach IoU 0.5 with any anchor, so they get too few positives
- `retina_focal` – loss modification on top of the best of the above: focal γ = 1.5, α = 0.5
  (less down-weighting of easy examples, higher weight on the rare positives)

In [ ]:
RUNS = {
    "retina_base": {},
    "retina_anchors": {**DATA_ANCHORS},
    "retina_anchors_iou": {**DATA_ANCHORS, "fg_iou": 0.4, "bg_iou": 0.3},
}

histories = {}


def run(name, changes):
    seed_everything(42)
    cfg = {**BASE, **changes}
    print(f"\n=== {name}: {changes}")
    model = build_retinanet(cfg, pretrained=not SMOKE)
    histories[name] = train_detector(model, train_frames, val_frames, cfg, DEVICE, OUT / f"{name}.pt")
    preds, ms = predict(model, val_frames, DEVICE)
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    metrics = evaluate(val_frames, preds)
    log_result(name, metrics, {"family": "one-stage anchor", "ms_per_img": round(ms, 1),
                               "train_min": round(sum(h["train_min"] for h in histories[name]), 1),
                               "config": str({k: v for k, v in changes.items() if k not in DATA_ANCHORS} |
                                             ({"anchors": "data"} if "anchor_sizes" in changes else {}))})
    print(name, metrics, f"{ms:.1f} ms/img")
    del model
    torch.cuda.empty_cache()
    return metrics


for name, changes in RUNS.items():
    run(name, changes)

# focal-loss run on top of the better anchor setting
results = pd.read_csv(OUT / "results.csv")
best_prev = results[results.experiment.isin(RUNS)].sort_values("mAP").experiment.iloc[-1]
RUNS["retina_focal"] = {**RUNS[best_prev], "focal_gamma": 1.5, "focal_alpha": 0.5}
run("retina_focal", RUNS["retina_focal"])

## Curves and comparison with the tuned two-stage detector

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    h = pd.DataFrame(h)
    axes[0].plot(h.epoch, h.mAP, marker="o", label=name)
    axes[1].plot(h.epoch, h.AP_small, marker="o", label=name)
axes[0].set_title("val mAP@[.5:.95]")
axes[1].set_title("val AP_small")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures_retina_curves.png", dpi=120)
plt.show()

results = pd.read_csv(OUT / "results.csv")
cols = ["experiment", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR", "ms_per_img", "train_min"]
results[results.experiment.str.startswith(("frcnn", "retina"))][cols]

## Recall by size and error types: best RetinaNet vs best Faster R-CNN

In [ ]:
from common import load_preds

best_retina = results[results.experiment.str.startswith("retina")].sort_values("mAP").experiment.iloc[-1]
best_frcnn = results[results.experiment.str.startswith("frcnn")].sort_values("mAP").experiment.iloc[-1]
table = {}
for name in [best_frcnn, best_retina]:
    p = load_preds(OUT / f"preds_val_{name}.npz")
    gts, fps = error_breakdown(val_frames, p, score_thr=0.3)
    table[name] = recall_by_size(gts)
    _, _, fp_types = summarise_errors(gts, fps)
    print(name, "false positives:", fp_types.to_dict())
pd.DataFrame(table)

TODO (your analysis):
- did the data-driven anchors help, and for which sizes?
- what problems come from removing the proposal stage (foreground/background imbalance over
  ~100k+ anchors per image, no RoI re-sampling, no second refinement) and how focal loss /
  IoU thresholds address them — support with the numbers above.